# 🚀 Super-Fast Multi-Agent Fusion Training (1,000 Samples)

This notebook performs end-to-end inference for all 4 agents on a small subset of 1,000 samples and trains the final Meta-Classifier. 

## Agents Included:
1.  **Spectral Agent** (XGBoost from Drive)
2.  **Prosodic Agent** (XGBoost from Drive)
3.  **Linguistic Agent** (Whisper-Tiny + Fine-tuned BERT from Drive)
4.  **SSL Agent** (JYP2024 WavLM-Base-Pruning from HuggingFace)

## Strategy:
*   **Speed over Quantity**: We process only 1,000 samples (500 Bonafide / 500 Spoof) to get the fusion model ready in minutes.
*   **Inference-Only**: We use the pre-trained/fine-tuned agents to get probability scores.
*   **Meta-Training**: We train a **Logistic Regression** meta-model on these scores.

In [ ]:
# Install dependencies for all agents
!pip install -q xgboost transformers datasets librosa soundfile imbalanced-learn joblib

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from google.colab import drive
import random

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Project Paths
# ADJUST 'PROJECT_ROOT' if your folder name is different in MyDrive
PROJECT_ROOT = Path("/content/drive/MyDrive/Multi-Agent-Detection-of-AI-Generated-Speech")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DRIVE_DATA = Path("/content/drive/MyDrive/40_PER_22_Data")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Directory: {DRIVE_DATA}")

In [ ]:
# ────────────────────────
# 3. Initialize All Agents
# ────────────────────────
from spectral.spectral_model import SpectralAgent
from prosodic.prosodic_model import ProsodicAgent
from linguistic.linguistic_model import LinguisticAgent
from ssl.ssl_model import SSLAgent

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing Agents on {device}...")

# Paths from your Drive screenshot
spec_agent = SpectralAgent(DRIVE_DATA / "spectral_results")
pros_agent = ProsodicAgent(DRIVE_DATA / "prosodic_results")
ling_agent = LinguisticAgent(DRIVE_DATA / "linguistic_bert_model", device=device)
ssl_agent  = SSLAgent("JYP2024/Wedefense_ASV2025_WavLM_Base_Pruning", device=device)

print("✅ All agents initialized!")

In [ ]:
# ────────────────────
# 4. Sample 1,000 Audio Files
# ────────────────────
# Adjust DATA_PATH to where your train audio is stored
DATA_PATH = DRIVE_DATA / "train"

all_samples = []
for label, sub in [(0, "bonafide"), (1, "spoof")]:
    folder = DATA_PATH / sub
    if folder.exists():
        files = list(folder.glob("*.flac")) + list(folder.glob("*.wav"))
        sampled = random.sample(files, min(len(files), 500)) # 500 per class
        for f in sampled:
            all_samples.append({'path': f, 'label': label})

random.shuffle(all_samples)
print(f"[*] Target samples: {len(all_samples)}")

In [ ]:
# ────────────────────
# 5. Run End-to-End Inference
# ────────────────────
results = []

for item in tqdm(all_samples, desc="Agent Inference"):
    path = item['path']
    try:
        # Probability scores (1.0 = Spoof)
        p_spec = spec_agent.predict(path)
        p_pros = pros_agent.predict(path)
        p_ling = ling_agent.predict(path)
        p_ssl  = ssl_agent.predict(path)
        
        results.append({
            'P_spec': p_spec,
            'P_pros': p_pros,
            'P_ling': p_ling,
            'P_ssl': p_ssl,
            'label': item['label']
        })
    except Exception as e:
        print(f"[!] Error on {path.name}: {e}")

fusion_df = pd.DataFrame(results)
fusion_df.to_csv(DRIVE_DATA / "fusion_1k_data.csv", index=False)
print(f"✅ Inference Done. Data shape: {fusion_df.shape}")

In [ ]:
# ────────────────────
# 6. Train Meta-Classifier
# ────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

X = fusion_df[['P_spec', 'P_pros', 'P_ling', 'P_ssl']].values
y = fusion_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(X_train, y_train)

print("\n─── Meta-Agent Trust Weights ───")
for name, coef in zip(['Spectral', 'Prosodic', 'Linguistic', 'SSL'], meta_model.coef_[0]):
    print(f"{name:>10}: {coef:+.4f}")

# Evaluation
y_prob = meta_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("\n─── Final Fusion Results ───")
print(classification_report(y_test, y_pred))
print(f"Final Multi-Agent AUC: {roc_auc_score(y_test, y_prob):.4f}")

joblib.dump(meta_model, DRIVE_DATA / "fusion_meta_model.pkl")
print(f"✅ Saved Meta-Model to {DRIVE_DATA / 'fusion_meta_model.pkl'}")